In [61]:
from google.colab import files
uploaded=files.upload()

Saving 1_TestSet_3Class.csv to 1_TestSet_3Class (3).csv
Saving 1_TrainingSet_3Class.csv to 1_TrainingSet_3Class (3).csv


In [62]:
import nltk

# Download necessary NLTK resources
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")
nltk.download("wordnet")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

**Preprocessing**

In [63]:
import pandas as pd


In [64]:
df = pd.read_csv('1_TrainingSet_3Class.csv', header=None)
df.columns = ['label', 'id', 'text']
display(df.head())

,label,id,text
0,2,s801810168862408705,UPDATE: Rt. in York County has reopened near C...
1,0,s899723031702065152,#Fargo #ND #USA - Sr. Clinical Research Associ...
2,0,s840600120727130112,Just two of the biggest fans riding the ride i...
3,1,s900335719461400580,Wreck on Southbound just north of Atlanta High...
4,0,s657684246153314304,Justin bieber's new album is going to be amazi...


In [65]:
df.columns = ['label', 'id', 'text']

In [66]:
import re
from bs4 import BeautifulSoup
import nltk


In [67]:
# Convert the text column to lowercase
df['text'] = df['text'].astype(str).str.lower()



In [68]:
def remove_html_tags(text):
    # Ensure the input is treated as a string before passing to BeautifulSoup
    soup = BeautifulSoup(str(text), "html.parser")
    return soup.get_text()

df['text'] = df['text'].apply(remove_html_tags)

In [69]:
def remove_extra_whitespace(text):
    return re.sub(r"\s+", " ", text).strip()

df['text'] = df['text'].apply(remove_extra_whitespace)

In [70]:
def remove_urls(text):
    return re.sub(r"http\S+|www\S+|https\S+", "", text, flags=re.MULTILINE)

df['text'] = df['text'].apply(remove_urls)

In [71]:
def remove_punctuation_and_digits(text):
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\d+", "", text)
    return text

df['text'] = df['text'].apply(remove_punctuation_and_digits)

In [72]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

def remove_stopwords(text):
    stop_words = set(stopwords.words("english"))
    tokens = word_tokenize(text)
    filtered_tokens = [token for token in tokens if token.lower() not in stop_words]
    return " ".join(filtered_tokens)

df['text'] = df['text'].apply(remove_stopwords)

In [73]:
def tokenize_text(text):
    return nltk.word_tokenize(text)

df['text'] = df['text'].apply(tokenize_text)

In [ ]:
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

def lemmatize_text(tokens_list):
    # The 'text' column now contains lists of tokens, not single strings.
    # We iterate directly over the list to lemmatize each token.
    lemmatized = [lemmatizer.lemmatize(word) for word in tokens_list]
    return " ".join(lemmatized)

df['text'] = df['text'].apply(lemmatize_text)

In [ ]:
import pandas as pd

test_df = pd.read_csv('1_TestSet_3Class.csv', header=None)
test_df.columns = ['label', 'id', 'text']

display(test_df.head())
test_df['text'] = test_df['text'].astype(str).str.lower()

test_df['text'] = test_df['text'].apply(remove_html_tags)
test_df['text'] = test_df['text'].apply(remove_extra_whitespace)
test_df['text'] = test_df['text'].apply(remove_urls)
test_df['text'] = test_df['text'].apply(remove_punctuation_and_digits)
test_df['text'] = test_df['text'].apply(remove_stopwords)
test_df['text'] = test_df['text'].apply(tokenize_text)
test_df['text'] = test_df['text'].apply(lemmatize_text)

In [ ]:
# Save back to CSV
df.to_csv("1_TrainingSet_3Class_lower.csv", index=False)

In [ ]:
display(df[['text']].head(10))

In [ ]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(df)
test_dataset = Dataset.from_pandas(test_df)

In [ ]:
from transformers import AutoTokenizer

def tokenize_function(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

In [74]:
# Prompt examples with true labels
samples = [
    {
        "tweet": "Massive traffic jam after a car accident on highway",
        "label": 0
    },
    {
        "tweet": "Road closed due to construction work downtown",
        "label": 1
    },
    {
        "tweet": "Traffic is moving smoothly this morning",
        "label": 2
    }
]

print("Prompt Testing BEFORE Fine-Tuning\n")

correct = 0

for sample in samples:

    tweet = sample["tweet"]
    true_label = sample["label"]

    prompt = f"""
Classify the traffic tweet into one category:
0 = accident
1 = construction
2 = normal traffic

Tweet: {tweet}
"""

    prediction = classifier(tweet)[0]

    predicted_label = int(prediction["label"].split("_")[1])

    verdict = "correct" if predicted_label == true_label else "incorrect"

    if verdict == "correct":
        correct += 1

    print("="*70)
    print("Prompt:")
    print(prompt)

    print("Predicted Label:", predicted_label)
    print("Confidence:", prediction["score"])
    print("True Label:", true_label)
    print("Verdict:", verdict)

# Accuracy
accuracy = correct / len(samples)

print("\nOverall Accuracy BEFORE Fine-Tuning:", accuracy)

Prompt Testing BEFORE Fine-Tuning

Prompt:

Classify the traffic tweet into one category:
0 = accident
1 = construction
2 = normal traffic

Tweet: Massive traffic jam after a car accident on highway

Predicted Label: 1
Confidence: 0.3644564747810364
True Label: 0
Verdict: incorrect
Prompt:

Classify the traffic tweet into one category:
0 = accident
1 = construction
2 = normal traffic

Tweet: Road closed due to construction work downtown

Predicted Label: 1
Confidence: 0.3547247350215912
True Label: 1
Verdict: correct
Prompt:

Classify the traffic tweet into one category:
0 = accident
1 = construction
2 = normal traffic

Tweet: Traffic is moving smoothly this morning

Predicted Label: 1
Confidence: 0.34805822372436523
True Label: 2
Verdict: incorrect

Overall Accuracy BEFORE Fine-Tuning: 0.3333333333333333


Fine tuning

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from peft import LoraConfig, TaskType, get_peft_model


# Base model with classification head
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3
)

# Apply LoRA adapters
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["q_lin", "v_lin"]
)
 #Wrap the model with LoRA.
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Training arguments (cleaned up)
args = TrainingArguments(
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    output_dir = './results',
    num_train_epochs=20,
    save_steps=1000,
    save_total_limit=2
)

trainer = Trainer(model=model,
                  args=args,
                  train_dataset=tokenized_train,
                  eval_dataset =  tokenized_test,
                 )

# Fine-tune
trainer.train()

# Save final model
model.save_pretrained("final_lora_distilbert_model")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 740,355 || all params: 67,696,134 || trainable%: 1.0936


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


In [ ]:
# -------------------------------
# Prompt evaluation AFTER fine-tuning
# -------------------------------

from transformers import pipeline

# Use the fine-tuned model
classifier_after = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer
)

samples = [
    {
        "tweet": "Massive traffic jam after a car accident on highway",
        "label": 0
    },
    {
        "tweet": "Road closed due to construction work downtown",
        "label": 1
    },
    {
        "tweet": "Traffic is moving smoothly this morning",
        "label": 2
    }
]

print("Prompt Testing AFTER Fine-Tuning\n")

correct = 0

for sample in samples:

    tweet = sample["tweet"]
    true_label = sample["label"]

    prediction = classifier_after(tweet)[0]

    predicted_label = int(prediction["label"].split("_")[1])

    verdict = "correct" if predicted_label == true_label else "incorrect"

    if verdict == "correct":
        correct += 1

    print("="*70)
    print("Tweet:", tweet)

    print("Predicted Label:", predicted_label)
    print("Confidence:", prediction["score"])
    print("True Label:", true_label)
    print("Verdict:", verdict)

accuracy = correct / len(samples)

print("\nOverall Accuracy AFTER Fine-Tuning:", accuracy)